# PassWard: Improved Password-Strength Training

This notebook reproduces the current SVM training pipeline in an easier-to-follow format. It creates the model files used by the Flask application.


## 1. Import libraries

These libraries load the dataset, create text features, train the classifier, and measure performance.


In [ ]:
import pickle
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import FeatureUnion
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


## 2. Load and clean the dataset

Each row contains a password and its training label. Invalid rows and missing values are removed, and both columns are converted to the expected types.


In [ ]:
data = pd.read_csv('data.csv', on_bad_lines='skip')
data = data.dropna(subset=['password', 'strength'])
data['password'] = data['password'].astype(str)
data['strength'] = data['strength'].astype(int)
print(data.shape)
data.head()


## 3. Visualize the data before training

These charts help us understand the cleaned dataset before the model is trained. They do not change the training data.


### Graph 1: Class distribution bar plot


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plot_data = data.copy()
plot_data['length'] = plot_data['password'].str.len()
plot_data['has_lowercase'] = plot_data['password'].str.contains(r'[a-z]', regex=True)
plot_data['has_uppercase'] = plot_data['password'].str.contains(r'[A-Z]', regex=True)
plot_data['has_number'] = plot_data['password'].str.contains(r'[0-9]', regex=True)
plot_data['has_special'] = plot_data['password'].str.contains(r'[^A-Za-z0-9]', regex=True)

plt.figure(figsize=(8, 5))
plot_data['strength'].value_counts().sort_index().plot(kind='bar', color=['#d85c67', '#d09a20', '#169b72'])
plt.title('Class distribution - bar plot')
plt.xlabel('Strength label')
plt.ylabel('Number of passwords')
plt.tight_layout()
plt.show()


### Graph 2: Password length box plot


In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(data=plot_data, x='strength', y='length', palette=['#d85c67', '#d09a20', '#169b72'], hue='strength', legend=False)
plt.title('Password length by class - box plot')
plt.xlabel('Strength label')
plt.ylabel('Character length')
plt.tight_layout()
plt.show()

type_rates = plot_data.groupby('strength')[['has_lowercase', 'has_uppercase', 'has_number', 'has_special']].mean()


### Graph 3: Character-type grouped bar plot


In [ ]:
plt.figure(figsize=(8, 5))
type_rates.plot(kind='bar', color=['#65a7ff', '#5ce1b5', '#d09a20', '#d85c67'])
plt.title('Character types by class - grouped bar plot')
plt.xlabel('Strength label')
plt.ylabel('Proportion of passwords')
plt.legend(['Lowercase', 'Uppercase', 'Numbers', 'Special'])
plt.tight_layout()
plt.show()


### Graph 4: Correlation heatmap


In [ ]:
correlation_data = plot_data[['length', 'has_lowercase', 'has_uppercase', 'has_number', 'has_special']].copy()
correlation_data['unique_characters'] = plot_data['password'].apply(lambda value: len(set(value)))

plt.figure(figsize=(8, 6))
sns.heatmap(correlation_data.corr(), annot=True, cmap='YlGnBu', vmin=-1, vmax=1, fmt='.2f')
plt.title('Correlation heatmap of password characteristics')
plt.tight_layout()
plt.show()


## 4. Split training and testing data

The data is split into training and testing portions. Stratification keeps the class proportions similar in both portions.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    data['password'], data['strength'], test_size=0.2,
    random_state=42, stratify=data['strength']
)
print(len(X_train), len(X_test))


## 5. Extract character-level TF-IDF features

Passwords are strings, so character n-grams are used instead of word features. They capture length patterns, substitutions, repeated characters, and character combinations.


In [ ]:
features = FeatureUnion([
    ('char', TfidfVectorizer(analyzer='char', ngram_range=(2, 5), min_df=2, sublinear_tf=True)),
    ('char_wb', TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 5), min_df=2, sublinear_tf=True)),
])
X_train_vec = features.fit_transform(X_train)
X_test_vec = features.transform(X_test)
print(X_train_vec.shape, X_test_vec.shape)


## 6. Train the balanced Linear SVM

Balanced class weights reduce the effect of the uneven class distribution in the CSV.


In [ ]:
model = LinearSVC(class_weight='balanced', C=2.0, random_state=42)
model.fit(X_train_vec, y_train)
predicted = model.predict(X_test_vec)


## 7. Evaluate the model

Accuracy, confusion matrix, precision, recall, and F1-score show how well the classifier performs on unseen passwords.


In [ ]:
print(f'Accuracy: {accuracy_score(y_test, predicted):.4f}')
print('Confusion matrix:')
print(confusion_matrix(y_test, predicted))
print(classification_report(y_test, predicted, digits=4))


## 8. Visualize model performance

These charts summarize performance on the held-out test set after training. They do not retrain or change the model.


In [ ]:
# Graph 5: Confusion matrix showing correct and incorrect predictions by class.
import matplotlib.pyplot as plt
import seaborn as sns

labels = sorted(y_test.unique())
cm = confusion_matrix(y_test, predicted, labels=labels)
plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels)
plt.title('Confusion matrix')
plt.xlabel('Predicted strength')
plt.ylabel('Actual strength')
plt.tight_layout()
plt.show()

# Graph 6: Precision, recall, and F1-score for each strength class.
report = classification_report(y_test, predicted, labels=labels, output_dict=True, zero_division=0)
metric_table = pd.DataFrame({label: [report[str(label)]['precision'], report[str(label)]['recall'], report[str(label)]['f1-score']] for label in labels}, index=['Precision', 'Recall', 'F1-score'])
metric_table.T.plot(kind='bar', figsize=(8, 5), color=['#65a7ff', '#5ce1b5', '#d09a20'])
plt.title('Precision, recall, and F1-score by class')
plt.xlabel('Strength class')
plt.ylabel('Score')
plt.ylim(0, 1.05)
plt.legend(title='Metric')
plt.tight_layout()
plt.show()


## 9. Save the artifacts

The Flask app loads these two files. The model predicts the class, while the feature object converts new passwords into the same TF-IDF representation used during training.


In [ ]:
with open('password_strength_svm.pkl', 'wb') as f:
    pickle.dump(model, f)
with open('password_strength_features.pkl', 'wb') as f:
    pickle.dump(features, f)
print('Saved password_strength_svm.pkl and password_strength_features.pkl')
